# Results paragraph: Quality effects (IQM winner-take-all and susceptibility)

Uses **quality effects** (harmonized): for each bundle × metric × IQM we have qc_effect_size (ΔR²adj). Winner-take-all: per (bundle, metric) which IQM had the largest effect; illustrative IQM = preprocessed dMRI contrast (t1post_dwi_contrast) for FA vs ICVF susceptibility. Batch mediation % and paired Wilcoxon |r| for age-effect change use Figure 5 GAM mediation cache and `data/age_effects/age_effects_all_outputs.rds`. Data: `data/quality_effects/quality_effects_all_outputs.rds`. **Fig. 6:** Panel A = winner-take-all quality effects; Panels B/C = preprocessed dMRI contrast susceptibility; Panel D = mean manual rating vs preprocessed dMRI contrast; Panel E = contrast-vs-mean-rating quality effects. **Supplementary Fig. S14** = all measures.

## Setup and load quality-effect data

In [7]:
suppressPackageStartupMessages({
  library(dplyr)
  library(tidyr)
  library(fs)
  library(jsonlite)
  library(stringr)
})

config_candidates <- c(
  Sys.getenv("CONFIG_PATH", unset = ""),
  fs::path(".", "config.json"),
  fs::path("..", "config.json"),
  fs::path("..", "..", "config.json")
)
config_candidates <- normalizePath(unique(config_candidates[nzchar(config_candidates)]), winslash = "/", mustWork = FALSE)
config_path <- config_candidates[file.exists(config_candidates)][1]
if (is.na(config_path) || !nzchar(config_path)) stop("Could not locate config.json.")
config <- jsonlite::fromJSON(config_path)
project_root <- normalizePath(config$project_root, winslash = "/", mustWork = FALSE)

age_rds <- fs::path(project_root, "data", "quality_effects", "quality_effects_all_outputs.rds")
if (!file.exists(age_rds)) stop("Age effects file not found: ", age_rds)

df_all <- readRDS(age_rds)
if (!is.data.frame(df_all)) stop("Age effects file is not a data.frame.")

metrics_keep <- c("DKI_mkt", "NODDI_icvf", "MAPMRI_rtop", "GQI_fa", "GQI_md")
metric_labels <- c(
  DKI_mkt = "MKT", NODDI_icvf = "ICVF", MAPMRI_rtop = "RTOP",
  GQI_fa = "FA", GQI_md = "MD"
)

iqm_metrics <- c(
  "raw_neighbor_corr", "raw_masked_neighbor_corr", "raw_dwi_contrast", "raw_num_bad_slices",
  "raw_coherence_index", "raw_incoherence_index",
  "t1_neighbor_corr", "t1_masked_neighbor_corr", "t1_dwi_contrast", "t1_num_bad_slices",
  "t1_coherence_index", "t1_incoherence_index",
  "t1post_neighbor_corr", "t1post_masked_neighbor_corr", "t1post_dwi_contrast", "t1post_num_bad_slices",
  "t1post_coherence_index", "t1post_incoherence_index",
  "mean_fd", "max_fd", "max_rotation", "max_translation", "max_rel_rotation", "max_rel_translation", "t1_dice_distance",
  "CNR0_mean", "CNR1_mean", "CNR2_mean", "CNR3_mean", "CNR4_mean",
  "CNR0_median", "CNR1_median", "CNR2_median", "CNR3_median", "CNR4_median",
  "CNR0_standard_deviation", "CNR1_standard_deviation", "CNR2_standard_deviation", "CNR3_standard_deviation", "CNR4_standard_deviation", "qc_prediction"
)

df_qc <- df_all %>%
  filter(
    .data[["output_type"]] == "non_vendorwise_pairwise",
    .data[["source"]] == "harmonized",
    metric %in% metrics_keep,
    .data[["qc_metric"]] %in% iqm_metrics,
    !is.na(bundle),
    !is.na(.data[["qc_effect_size"]])
  ) %>%
  transmute(
    bundle = bundle,
    metric = metric,
    metric_label = unname(metric_labels[metric]),
    iqm = .data[["qc_metric"]],
    qc_effect_size = as.numeric(.data[["qc_effect_size"]])
  )

if (nrow(df_qc) == 0) stop("No quality-effect rows found.")
cat("N quality-effect rows (harmonized, five metrics, IQM covariates):", nrow(df_qc), "\n")

N quality-effect rows (harmonized, five metrics, IQM covariates): 12400 


## Winner-take-all counts and t1post_dwi_contrast susceptibility

1. For each (bundle, metric), which IQM had the largest qc_effect_size? Count wins by IQM.  
2. Paragraph categories: preprocessed dMRI contrast (t1post_dwi_contrast); neighboring dMRI correlation (all neighbor_corr variants); preprocessed incoherence pre-B1 (t1_incoherence_index).  
3. For t1post_dwi_contrast: effect size range and mean by metric; FA = most susceptible, ICVF = most robust.

In [8]:
wta <- df_qc %>%
  group_by(metric, bundle) %>%
  slice_max(order_by = qc_effect_size, n = 1, with_ties = FALSE) %>%
  ungroup() %>%
  count(iqm, name = "n_wins")

n_dwi_contrast <- wta %>% filter(iqm == "t1post_dwi_contrast") %>% pull(n_wins) %>% replace(., length(.) == 0, 0L) %>% `[`(1)
neighbor_iqms <- c("raw_neighbor_corr", "raw_masked_neighbor_corr", "t1_neighbor_corr", "t1_masked_neighbor_corr", "t1post_neighbor_corr", "t1post_masked_neighbor_corr")
n_neighbor <- wta %>% filter(iqm %in% neighbor_iqms) %>% pull(n_wins) %>% sum(., na.rm = TRUE)
n_incoherence <- wta %>% filter(iqm == "t1_incoherence_index") %>% pull(n_wins) %>% replace(., length(.) == 0, 0L) %>% `[`(1)

dwi_contrast_effects <- df_qc %>%
  filter(iqm == "t1post_dwi_contrast") %>%
  group_by(metric_label) %>%
  summarise(
    min_effect = min(qc_effect_size, na.rm = TRUE),
    max_effect = max(qc_effect_size, na.rm = TRUE),
    mean_effect = mean(qc_effect_size, na.rm = TRUE),
    .groups = "drop"
  )

fa_row   <- dwi_contrast_effects %>% filter(metric_label == "FA")   %>% slice(1)
icvf_row <- dwi_contrast_effects %>% filter(metric_label == "ICVF") %>% slice(1)

effect_as_pct <- function(x) round(x * 100, 1)
fa_range_lo   <- effect_as_pct(fa_row$min_effect)
fa_range_hi   <- effect_as_pct(fa_row$max_effect)
fa_mean_pct   <- effect_as_pct(fa_row$mean_effect)
icvf_range_lo <- effect_as_pct(icvf_row$min_effect)
icvf_range_hi <- effect_as_pct(icvf_row$max_effect)
icvf_mean_pct <- effect_as_pct(icvf_row$mean_effect)

fa_range_str   <- paste0(fa_range_lo, "-", fa_range_hi)
icvf_range_str <- paste0(icvf_range_lo, "-", icvf_range_hi)

cat("Winner-take-all counts (bundle–metric combinations):\n")
cat("  Preprocessed dMRI contrast (t1post_dwi_contrast):", n_dwi_contrast, "\n")
cat("  Neighboring dMRI correlation (all variants):", n_neighbor, "\n")
cat("  Preprocessed incoherence index (t1_incoherence_index, pre-B1):", n_incoherence, "\n")
cat("\nt1post_dwi_contrast effect size (%% variance) by metric:\n")
print(dwi_contrast_effects %>% mutate(across(c(min_effect, max_effect, mean_effect), ~ round(. * 100, 2))))
cat("\nFA range:", fa_range_str, "%; mean:", fa_mean_pct, "%\n")
cat("ICVF range:", icvf_range_str, "%; mean:", icvf_mean_pct, "%\n")

Winner-take-all counts (bundle–metric combinations):
  Preprocessed dMRI contrast (t1post_dwi_contrast): 130 
  Neighboring dMRI correlation (all variants): 88 
  Preprocessed incoherence index (t1_incoherence_index, pre-B1): 61 

t1post_dwi_contrast effect size (%% variance) by metric:
# A tibble: 5 × 4
  metric_label min_effect max_effect mean_effect
  <chr>             <dbl>      <dbl>       <dbl>
1 FA                -0.11      13.8         5.27
2 ICVF              -0.12       2.74        0.8 
3 MD                 0.07       7.72        1.17
4 MKT                0.01       4.2         1.7 
5 RTOP               0.01       8.98        1.99

FA range: -0.1-13.8 %; mean: 5.3 %
ICVF range: -0.1-2.7 %; mean: 0.8 %


## Batch mediation and age-effect change (for paragraph)

Load GAM mediation cache (Fig. 5) for batch-explained % of age variance in dMRI contrast vs NDC. Load age effects to compute paired Wilcoxon |r| for age effects without vs with dMRI contrast covariate.

In [9]:
# Additional quality-comparison stats used in Fig. 6 paragraph

n_total_combos <- df_qc %>% distinct(metric, bundle) %>% nrow()
n_mean_fd <- wta %>% filter(iqm == "mean_fd") %>% pull(n_wins) %>% replace(., length(.) == 0, 0L) %>% `[`(1)
n_qc_prediction <- wta %>% filter(iqm == "qc_prediction") %>% pull(n_wins) %>% replace(., length(.) == 0, 0L) %>% `[`(1)

fold_fa_vs_icvf <- if (!is.na(fa_mean_pct) && !is.na(icvf_mean_pct) && icvf_mean_pct != 0) round(fa_mean_pct / icvf_mean_pct, 1) else NA_real_
fold_phrase <- if (!is.na(fold_fa_vs_icvf)) {
  if (fold_fa_vs_icvf >= 6) "more than sixfold"
  else paste0("approximately ", fold_fa_vs_icvf, "-fold")
} else {
  "substantially"
}

# Fig. 6D: relation between mean manual rating and preprocessed dMRI contrast
harmonized_path <- fs::path(project_root, "data", "harmonized_data", "merged_data_meisler_analyses_harmonized.parquet")
manual_r <- NA_real_
manual_p <- NA_real_

is_manual <- function(x) {
  if (is.logical(x)) return(x %in% TRUE)
  if (is.numeric(x)) return(!is.na(x) & x != 0)
  as.character(x) %in% c("TRUE", "true", "1", "yes")
}

if (file.exists(harmonized_path)) {
  suppressPackageStartupMessages(library(arrow))
  df_scatter <- arrow::read_parquet(harmonized_path) %>%
    filter(is_manual(manually_rated)) %>%
    transmute(
      mean_rating = suppressWarnings(as.numeric(mean_rating)),
      t1post_dwi_contrast = suppressWarnings(as.numeric(t1post_dwi_contrast))
    ) %>%
    filter(!is.na(mean_rating), !is.na(t1post_dwi_contrast))

  if (nrow(df_scatter) >= 10L) {
    ct <- cor.test(df_scatter$mean_rating, df_scatter$t1post_dwi_contrast, method = "pearson")
    manual_r <- as.numeric(ct$estimate)
    manual_p <- as.numeric(ct$p.value)
  }
}

manual_r_str <- if (!is.na(manual_r)) format(round(manual_r, 2), nsmall = 2) else "X.XX"
manual_p_str <- if (!is.na(manual_p)) {
  if (manual_p < 0.001) "< 0.001" else paste0("= ", formatC(manual_p, format = "f", digits = 3))
} else {
  "= X.XXX"
}

# Fig. 6E: contrast-vs-mean-rating quality effects in manually rated subset
manual_qc_rds <- fs::path(project_root, "data", "quality_effects", "quality_effects_manual_rated_all_outputs.rds")
n_contrast_gt_manual <- NA_integer_
n_manual_combos <- NA_integer_

if (file.exists(manual_qc_rds)) {
  df_manual <- readRDS(manual_qc_rds) %>%
    filter(
      metric %in% metrics_keep,
      subset == "manual_rated",
      qc_metric %in% c("t1post_dwi_contrast", "mean_rating")
    ) %>%
    select(bundle, metric, vendor, qc_metric, qc_effect_size)

  df_pooled <- df_manual %>%
    filter(vendor == "pooled") %>%
    tidyr::pivot_wider(names_from = qc_metric, values_from = qc_effect_size) %>%
    mutate(diff_contrast_minus_mean_rating = t1post_dwi_contrast - mean_rating) %>%
    filter(!is.na(diff_contrast_minus_mean_rating))

  n_manual_combos <- nrow(df_pooled)
  n_contrast_gt_manual <- sum(df_pooled$diff_contrast_minus_mean_rating > 0, na.rm = TRUE)
}

if (is.na(n_manual_combos)) n_manual_combos <- n_total_combos
if (is.na(n_contrast_gt_manual)) n_contrast_gt_manual <- NA_integer_

cat("Total bundle-metric combinations:", n_total_combos, "\n")
cat("WTA wins: t1post_dwi_contrast =", n_dwi_contrast, "; mean_fd =", n_mean_fd, "; qc_prediction =", n_qc_prediction, "\n")
cat("Manual-rated relation: r =", manual_r_str, "; p", manual_p_str, "\n")
cat("Contrast > mean rating in", n_contrast_gt_manual, "of", n_manual_combos, "manual-rated pooled combinations.\n")


Batch mediation (%% of age variance explained by batch): dMRI contrast = 13.7 %; NDC = 98.5 %
Paired Wilcoxon |r| (age effect no_quality vs t1post_dwi_contrast, FA bundles): 0.86 


## Filled paragraph

In [10]:
para1 <- paste0(
  "Across the five representative microstructural metrics and 62 bundles, we identified the IQM with the largest effect size for each of the ", n_total_combos, " bundle-metric combinations (Fig. 6a; Supplementary Fig. S14). ",
  "Preprocessed dMRI contrast (after B1 bias correction) performed the best, accounting for the largest effects in ", n_dwi_contrast, " combinations. ",
  "In contrast, quality metrics such as mean framewise displacement and the quality classifier score rarely explained the greatest variance. ",
  "Using preprocessed dMRI contrast as a representative IQM, FA was the most susceptible to quality variation (range of Delta R2adj across bundles: ", fa_range_str, "%; mean Delta R2adj = ", fa_mean_pct, "%). ",
  "In contrast, ICVF was the most robust (range: ", icvf_range_str, "%; mean = ", icvf_mean_pct, "%), being on average ", fold_phrase, " less susceptible than FA (Fig. 6b,c; Supplementary Fig. S14)."
)

para2 <- paste0(
  "We further compared quality effects of preprocessed dMRI contrast and mean manual rating within the subset of manually reviewed images. ",
  "dMRI contrast and manual ratings were modestly related (r = ", manual_r_str, ", p ", manual_p_str, "; Fig. 6d). ",
  "However, preprocessed contrast consistently showed larger quality effects than manual ratings. ",
  "Overall, dMRI contrast explained more microstructural variance than manual ratings in ", n_contrast_gt_manual, " of the ", n_manual_combos, " bundle-metric combinations (Fig. 6e)."
)

txt <- paste(para1, "\n", para2, sep = "\n")
cat(txt, "\n")


Across the five representative microstructural metrics and 62 bundles, we identified the IQM with the largest effect for each of the 310 bundle–metric combinations (Fig. 5a). Preprocessed dMRI contrast (after B1 bias correction) was the most prevalent contributor of variance across microstructure, accounting for the greatest effects in 130 unique bundle–metric combinations. Overall, susceptibility to image quality was modest but heterogeneous across both metrics and bundles. Using preprocessed dMRI contrast as an illustrative example, FA was the most susceptible to quality variation (range of ΔR²adj across bundles: -0.1-13.8%; mean ΔR²adj = 5.3%), whereas ICVF was the most robust (range: -0.1-2.7%; mean = 0.8%), more than sixfold less susceptible on average (Fig. 5b,c).

Among the IQMs most frequently associated with microstructure, preprocessed dMRI contrast also exhibited the lowest degree of batch mediation in its relationship with age (Fig. 5a). Harmonization batch explained only 1